<a href="https://colab.research.google.com/github/2303A52060/High-performace-computing/blob/main/HPC_ASS_6_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import multiprocessing as mp
from multiprocessing import shared_memory
import time
import os

# Configuration
N = 100_000_000  # 100 Million doubles (~800 MB)
NUM_WORKERS = mp.cpu_count()

def init_worker(shm_name, shape, dtype, start, end):
    """Worker to perform 'First Touch' initialization"""
    # Attach to existing shared memory
    shm = shared_memory.SharedMemory(name=shm_name)
    arr = np.ndarray(shape, dtype=dtype, buffer=shm.buf)

    # Write to memory (triggers physical page allocation on this NUMA node)
    arr[start:end] = 1.0

    shm.close()

def sum_worker(shm_name, shape, dtype, start, end, conn):
    """Worker to perform summation"""
    # Attach to existing shared memory
    shm = shared_memory.SharedMemory(name=shm_name)
    arr = np.ndarray(shape, dtype=dtype, buffer=shm.buf)

    # Compute local sum
    local_sum = np.sum(arr[start:end])

    shm.close()
    conn.send(local_sum)

def main():
    print(f"Starting NUMA First Touch Lab (Python)")
    print(f"Array Size: {N} elements ({N * 8 / 1024 / 1024:.2f} MB)")
    print(f"Workers: {NUM_WORKERS}")

    # 1. Allocate Memory (Virtual allocation, similar to malloc)
    # We use shared_memory to ensure processes share the same physical pages
    shm = shared_memory.SharedMemory(create=True, size=N * 8)
    arr = np.ndarray((N,), dtype=np.float64, buffer=shm.buf)

    # 2. First Touch Initialization (Parallel)
    # Split array into chunks for each worker
    chunk_size = N // NUM_WORKERS
    processes = []

    start_time = time.perf_counter()

    for i in range(NUM_WORKERS):
        start_idx = i * chunk_size
        end_idx = (i + 1) * chunk_size if i < NUM_WORKERS - 1 else N

        p = mp.Process(target=init_worker, args=(shm.name, (N,), np.float64, start_idx, end_idx))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    init_time = time.perf_counter() - start_time
    print(f"Initialization Time: {init_time:.4f} seconds")

    # 3. Summation (Parallel)
    processes = []
    parent_conns = []

    start_time = time.perf_counter()

    for i in range(NUM_WORKERS):
        start_idx = i * chunk_size
        end_idx = (i + 1) * chunk_size if i < NUM_WORKERS - 1 else N

        parent_conn, child_conn = mp.Pipe()
        parent_conns.append(parent_conn)

        p = mp.Process(target=sum_worker, args=(shm.name, (N,), np.float64, start_idx, end_idx, child_conn))
        p.start()
        processes.append(p)

    # Collect results
    total_sum = 0
    for conn in parent_conns:
        total_sum += conn.recv()

    for p in processes:
        p.join()

    comp_time = time.perf_counter() - start_time
    print(f"Computation Time: {comp_time:.4f} seconds")
    print(f"Sum = {total_sum:.1f}")

    # Cleanup
    shm.close()
    shm.unlink()

if __name__ == "__main__":
    main()

Starting NUMA First Touch Lab (Python)
Array Size: 100000000 elements (762.94 MB)
Workers: 2
Initialization Time: 0.5603 seconds
Computation Time: 0.1552 seconds
Sum = 100000000.0


In [2]:
import numpy as np
import time

N = 100_000_000

# 1. Memory Allocation and Initialization
# Using np.full for direct initialization with a specific value
a = np.full(N, 1.0, dtype=np.float32)
b = np.full(N, 2.0, dtype=np.float32)
c = np.zeros(N, dtype=np.float32) # Initialize c to zeros

print(f"Array Size: {N} elements ({N * 4 / (1024 * 1024):.2f} MB for float32)")

# 2. SIMD-like Vectorized Addition using NumPy
# NumPy operations are inherently vectorized and optimized, similar to OpenMP SIMD.
t0 = time.perf_counter()
c = a + b
t1 = time.perf_counter()
print(f"Vectorized Addition Time (NumPy): {t1 - t0:.6f} s")

# 3. Quick checksum to prevent optimizing-away
sum_c = np.sum(c)
print(f"Checksum: {sum_c:.1f}")

# Note: In Python, memory deallocation for NumPy arrays is handled automatically
# by the garbage collector when arrays are no longer referenced.

Array Size: 100000000 elements (381.47 MB for float32)
Vectorized Addition Time (NumPy): 0.088127 s
Checksum: 300000000.0


What is first-touch policy?

The "first-touch" policy in the context of NUMA (Non-Uniform Memory Access) refers to the strategy of allocating memory pages to the NUMA node where the first CPU thread or process to access that memory resides. When a program allocates a large chunk of memory, the operating system doesn't immediately assign physical pages to all of it. Instead, it waits until a specific part of that memory is accessed (a 'page fault' occurs). At that moment, the OS allocates a physical page on the NUMA node where the CPU making that first access is located. This policy is crucial for performance because it aims to place memory close to the CPU that will frequently use it, thereby maximizing local memory access and minimizing slower remote memory
 access.

Does NUMA affect correctness or performance?

NUMA primarily affects performance, not correctness. A program written for a non-NUMA system will still execute correctly on a NUMA system. However, its performance can vary dramatically. If data is frequently accessed by a CPU core that resides on a different NUMA node than where the data is physically located (remote access), it will incur higher latency and lower bandwidth, leading to slower execution. Conversely, if data is local to the accessing CPU (local access), performance will be much better. The goal of optimizing for NUMA is to improve performance by ensuring data and the threads that operate on it are co-located on the same NUMA node.

Why is memory bandwidth important in NUMA systems?

Memory bandwidth is critical in NUMA systems because it directly impacts how quickly CPUs can access data. In a NUMA architecture, each processor (or group of processors) has its own local memory controller and local bank of memory. This local memory path offers the highest possible bandwidth and lowest latency. When a CPU needs to access data located in the memory of another NUMA node (remote memory), the request must travel across an interconnect fabric to that remote node's memory controller and then back. This inter-node communication path has significantly lower bandwidth and higher latency compared to local memory access. Therefore, having sufficient memory bandwidth locally to each NUMA node is essential to feed the high-performance processors with data efficiently, and any reduction in effective bandwidth due to remote access can become a major bottleneck, limiting the scalability and performance of parallel applications.